# NBA Polymarket Backtest (Parquet Version) — Modal

Set `NBA_BOT_DATA_DIR`, `NBA_BOT_MODEL_PATH`, or `NBA_BOT_REPO_PATH` as needed before running.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PACKAGES = [
    'numpy<2.0.0',
    'pandas==2.2.2',
    'scikit-learn==1.5.2',
    'xgboost==2.1.3',
    'nba-on-court>=0.2.0,<1.0',
    'joblib>=1.3',
    'matplotlib>=3.8',
    'tqdm>=4.66',
    'polars>=1.0',
    'pyarrow>=16.0',
    'requests>=2.31',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)

REPO_URL = os.environ.get('NBA_BOT_REPO_URL', 'https://github.com/cyruslayo/nba_bot.git')
REPO_PATH = Path(os.environ.get('NBA_BOT_REPO_PATH', '/root/nba_bot')).expanduser()
DATA_DIR = Path(os.environ.get('NBA_BOT_DATA_DIR', '/root/data/nba_backtest')).expanduser()
MODEL_PATH = Path(os.environ.get('NBA_BOT_MODEL_PATH', str(DATA_DIR / 'xgb_model_t2.pkl'))).expanduser()

REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_PATH / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_PATH), 'pull', '--ff-only'], check=False)
elif REPO_PATH.exists():
    print(f'Repository path already exists and is not a git clone: {REPO_PATH}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'nba-bot'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_PATH), '--no-deps', '-q'], check=True)

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

print(f'Repo path: {REPO_PATH}')
print(f'Data directory: {DATA_DIR}')
print(f'Model path: {MODEL_PATH}')
print('Setup complete')

In [ ]:
import requests
from datetime import datetime, timedelta

def download_pmxt_parquet(start_date, end_date, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')

    while current <= end:
        date_str = current.strftime('%Y-%m-%d')
        for hour in range(24):
            hour_str = f'{hour:02d}'
            filename = f'polymarket_orderbook_{date_str}T{hour_str}.parquet'
            url = f'https://archive.pmxt.dev/Polymarket/{filename}'
            save_path = save_dir / filename

            if save_path.exists():
                continue

            print(f'Downloading {filename}...')
            response = requests.get(url, stream=True, timeout=60)
            if response.status_code != 200:
                continue

            with save_path.open('wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f'Saved to {save_path}')

        current += timedelta(days=1)

    print('Download process finished.')

# Example
# download_pmxt_parquet('2026-02-21', '2026-02-22', DATA_DIR)

In [ ]:
import json
import pandas as pd
import requests

def _safe_gamma_list(value):
    if isinstance(value, list):
        return value
    if value in (None, '', [], {}):
        return []
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def _as_utc_timestamp(value):
    ts = pd.to_datetime(value, utc=True, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    return ts

def _gamma_market_rows(events: list[dict]) -> list[dict]:
    rows = []
    for event in events:
        event_id = event.get('id')
        event_title = event.get('title')
        event_slug = event.get('slug')
        event_start_ts = _as_utc_timestamp(event.get('startDate'))
        for market in event.get('markets', []):
            outcomes = _safe_gamma_list(market.get('outcomes'))
            outcome_prices = _safe_gamma_list(market.get('outcomePrices'))
            clob_token_ids = _safe_gamma_list(market.get('clobTokenIds'))
            try:
                liquidity = float(market.get('liquidity', 0) or 0)
            except (TypeError, ValueError):
                liquidity = 0.0
            rows.append({
                'event_id': str(event_id) if event_id is not None else None,
                'event_title': event_title,
                'event_slug': event_slug,
                'event_start_ts': event_start_ts,
                'market_id': str(market.get('id')) if market.get('id') is not None else None,
                'question': market.get('question'),
                'outcomes': outcomes,
                'outcome_prices': outcome_prices,
                'clob_token_ids': clob_token_ids,
                'liquidity': liquidity,
                'active': market.get('active'),
                'closed': market.get('closed'),
            })
    return rows

def search_nba_markets(start_date: str, end_date: str, limit: int = 100, closed: bool = True, active: bool = True, buffer_days: int = 1) -> pd.DataFrame:
    start_bound = pd.Timestamp(start_date)
    end_bound = pd.Timestamp(end_date)
    if start_bound.tzinfo is None:
        start_bound = start_bound.tz_localize('UTC')
    else:
        start_bound = start_bound.tz_convert('UTC')
    if end_bound.tzinfo is None:
        end_bound = end_bound.tz_localize('UTC')
    else:
        end_bound = end_bound.tz_convert('UTC')
    start_bound = start_bound.normalize() - pd.Timedelta(days=buffer_days)
    end_bound = end_bound.normalize() + pd.Timedelta(days=buffer_days + 1) - pd.Timedelta(microseconds=1)

    status_filters = []
    if closed:
        status_filters.append({'active': 'false', 'closed': 'true'})
    if active:
        status_filters.append({'active': 'true', 'closed': 'false'})
    if not status_filters:
        status_filters.append({})

    seen_market_ids: set[str] = set()
    all_events: list[dict] = []

    for status_filter in status_filters:
        offset = 0
        while True:
            params = {
                'series_id': 10345,
                'tag_id': 100639,
                'limit': limit,
                'offset': offset,
            }
            params.update(status_filter)
            response = requests.get('https://gamma-api.polymarket.com/events', params=params, timeout=20)
            response.raise_for_status()
            page_events = response.json()
            if not page_events:
                break
            all_events.extend(page_events)
            if len(page_events) < limit:
                break
            offset += limit

    markets_df = pd.DataFrame(_gamma_market_rows(all_events))
    if markets_df.empty:
        return markets_df

    markets_df = markets_df[markets_df['market_id'].notna()].copy()
    markets_df['event_start_ts'] = pd.to_datetime(markets_df['event_start_ts'], utc=True, errors='coerce')
    markets_df = markets_df[markets_df['event_start_ts'].notna()].copy()
    markets_df = markets_df[(markets_df['event_start_ts'] >= start_bound) & (markets_df['event_start_ts'] <= end_bound)].copy()

    deduped_rows = []
    for row in markets_df.sort_values(['event_start_ts', 'market_id']).to_dict('records'):
        market_id = str(row['market_id'])
        if market_id in seen_market_ids:
            continue
        seen_market_ids.add(market_id)
        deduped_rows.append(row)

    return pd.DataFrame(deduped_rows).reset_index(drop=True)

In [ ]:
import ast
import json
import math
import re
from dataclasses import asdict, dataclass, field
from typing import Any

import matplotlib.pyplot as plt
import polars as pl
from tqdm.auto import tqdm

from nba_bot.features import STARTTYPE_FALLBACK
from nba_bot.model import load_model, predict_home_win_prob

PMXT_ROOT = Path(DATA_DIR)
PMXT_GLOB = '*.parquet'
INITIAL_BANKROLL = 1000.0
LATENCY_BUFFER_MS = 500
COOLDOWN_MS = 1500
MIN_TARGET_DELTA_USDC = 5.0
MIN_DIVERGENCE_RATIO = 0.10
MIN_RESTING_DEPTH_USDC = 50.0
ADVERSE_SELECTION_PENALTY = 0.01
FEE_RATE = 0.02
PER_GAME_BANKROLL_DIVISOR = 5.0
CHUNK_SIZE = 100000
MARKET_DATE_BUFFER_DAYS = 1
USE_ADVANCED_FEATURES = True
MARKET_ONLY_MODE = True
LOOKUP_CACHE_DIR = Path(DATA_DIR) / 'market_lookup_cache'
LOOKUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
BACKTEST_STATE_PATH = Path(DATA_DIR) / 'backtest_state.json'

MARKET_LOOKUP = pd.DataFrame()
MARKET_MATCHES = pd.DataFrame()
MATCH_DIAGNOSTICS: dict[str, pd.DataFrame] = {}

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
def build_game_catalog_from_gamma(markets_df: pd.DataFrame) -> pd.DataFrame:
    if markets_df.empty:
        return pd.DataFrame()

    catalog_rows = []
    seen_games = set()

    for row in markets_df.to_dict('records'):
        market_id = str(row.get('market_id') or '')
        event_title = str(row.get('event_title') or '')
        question = str(row.get('question') or '')
        event_start_ts = row.get('event_start_ts')
        outcomes = row.get('outcomes', [])
        clob_token_ids = row.get('clob_token_ids') or []

        if not market_id or not event_title:
            continue

        game_id = f'GAMMA_{market_id}'
        if game_id in seen_games:
            continue
        seen_games.add(game_id)

        home_team, away_team = None, None
        yes_team_side = 'home'

        title_lower = event_title.lower()
        question_lower = question.lower()

        if ' vs ' in title_lower:
            parts = title_lower.split(' vs ')
            if len(parts) == 2:
                away_team = parts[0].strip().title()
                home_team = parts[1].strip().title()
        elif ' vs ' in question_lower:
            parts = question_lower.split(' vs ')
            if len(parts) == 2:
                away_team = parts[0].strip().title()
                home_team = parts[1].strip().title()
        elif ' @ ' in title_lower:
            parts = title_lower.split(' @ ')
            if len(parts) == 2:
                away_team = parts[1].strip().title()
                home_team = parts[0].strip().title()

        if outcomes and len(outcomes) >= 2 and home_team:
            outcome_0 = str(outcomes[0]).lower()
            outcome_1 = str(outcomes[1]).lower()
            home_lower = home_team.lower()
            away_lower = away_team.lower() if away_team else ''

            if home_lower in outcome_0:
                yes_team_side = 'home'
            elif away_lower and away_lower in outcome_0:
                yes_team_side = 'away'
            elif home_lower in outcome_1:
                yes_team_side = 'away'
            elif away_lower and away_lower in outcome_1:
                yes_team_side = 'home'

        game_date = pd.Timestamp(event_start_ts).normalize() if pd.notna(event_start_ts) else None
        catalog_rows.append({
            'game_id': game_id,
            'game_date': game_date,
            'home_team': home_team or 'Unknown',
            'away_team': away_team or 'Unknown',
            'tipoff_time': event_start_ts,
            'market_id': market_id,
            'event_title': event_title,
            'question': question,
            'yes_team_side': yes_team_side,
            'liquidity': float(row.get('liquidity') or 0),
            'clob_token_ids': clob_token_ids,
        })

    return pd.DataFrame(catalog_rows)

def build_simple_timeline_from_gamma(markets_df: pd.DataFrame) -> pd.DataFrame:
    timeline_rows = []

    for row in markets_df.to_dict('records'):
        market_id = str(row.get('market_id') or '')
        event_start_ts = row.get('event_start_ts')

        if not market_id or pd.isna(event_start_ts):
            continue

        game_id = f'GAMMA_{market_id}'
        tipoff = pd.Timestamp(event_start_ts)
        timeline_rows.append({
            'game_id': game_id,
            'event_time': tipoff,
            'period': 1,
            'clock': '12:00',
            'home_score': 0.0,
            'away_score': 0.0,
            'time_remaining': 2880.0,
            'starttype_encoded': STARTTYPE_FALLBACK,
            'player_quality_home': 0.0,
            'player_quality_away': 0.0,
            'latent_time': tipoff + pd.to_timedelta(LATENCY_BUFFER_MS, unit='ms'),
            'tipoff_time': tipoff,
        })

    df = pd.DataFrame(timeline_rows)
    if not df.empty:
        df['latent_time'] = pd.to_datetime(df['latent_time'], utc=True).dt.as_unit('ns')
        df['tipoff_time'] = pd.to_datetime(df['tipoff_time'], utc=True).dt.as_unit('ns')
        df = df.sort_values(['game_id', 'latent_time']).reset_index(drop=True)
    return df

def join_ticks_with_timeline(frame: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    ticks = frame.copy()
    ticks['timestamp'] = pd.to_datetime(ticks['timestamp'], utc=True, errors='coerce').dt.as_unit('ns')
    ticks['game_id'] = ticks['game_id'].astype(str)
    ticks = ticks[ticks['timestamp'].notna()].sort_values('timestamp').reset_index(drop=True)

    timeline_fixed = timeline.copy()
    timeline_fixed['latent_time'] = pd.to_datetime(timeline_fixed['latent_time'], utc=True).dt.as_unit('ns')
    timeline_fixed = timeline_fixed.sort_values('latent_time').reset_index(drop=True)

    merged = pd.merge_asof(
        ticks,
        timeline_fixed,
        left_on='timestamp',
        right_on='latent_time',
        by='game_id',
        direction='backward',
        allow_exact_matches=True,
    )
    return merged[merged['timestamp'] >= merged['tipoff_time']].copy()

def _candidate_column(columns: list[str], names: list[str]) -> str | None:
    lowered = {str(c).lower(): c for c in columns}
    for name in names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    for column in columns:
        if any(name.lower() in str(column).lower() for name in names):
            return column
    return None

def attach_game_ids(frame: pd.DataFrame, market_lookup: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    key_col = _candidate_column(list(out.columns), ['token_id', 'asset_id', 'market_id', 'slug', 'event_slug', 'market'])
    if key_col is None:
        return pd.DataFrame()

    out['market_id_raw'] = out[key_col].astype(str)
    lookup_dict = dict(zip(market_lookup['lookup_key'].astype(str), market_lookup['game_id'].astype(str)))
    yes_team_dict = dict(zip(market_lookup['lookup_key'].astype(str), market_lookup['yes_team_side'].astype(str)))
    out['game_id'] = out['market_id_raw'].map(lookup_dict)
    out['yes_team_side'] = out['market_id_raw'].map(yes_team_dict)
    out['resolved_market_id'] = out['market_id_raw']
    return out[out['game_id'].notna()].copy()

def infer_pmxt_date_range(paths: list[Path]) -> tuple[str | None, str | None]:
    dates = set()
    pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
    for p in paths:
        match = pattern.search(p.name)
        if match:
            dates.add(match.group(1))
    if not dates:
        return None, None
    sorted_dates = sorted(dates)
    return sorted_dates[0], sorted_dates[-1]

def _safe_json(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except Exception:
            try:
                return ast.literal_eval(text)
            except Exception:
                return None
    return None

In [ ]:
def parse_levels(raw_levels: Any, descending: bool) -> list[tuple[float, float]]:
    payload = _safe_json(raw_levels)
    normalized = []
    for level in payload or []:
        price, size = None, None
        if isinstance(level, dict):
            price = level.get('price') or level.get('px') or level.get('p')
            size = level.get('size') or level.get('sz') or level.get('quantity') or level.get('q')
        elif isinstance(level, (list, tuple)) and len(level) >= 2:
            price, size = level[0], level[1]
        try:
            price, size = float(price), float(size)
        except (TypeError, ValueError):
            continue
        if price <= 0 or price >= 1 or size <= 0:
            continue
        normalized.append((float(price), float(size)))
    normalized.sort(key=lambda item: item[0], reverse=descending)
    return normalized

class OrderBookState:
    def __init__(self) -> None:
        self.bids: dict[float, float] = {}
        self.asks: dict[float, float] = {}

    def _apply_side(self, side: str, levels: list[tuple[float, float]], replace: bool) -> None:
        book = self.bids if side == 'bids' else self.asks
        if replace:
            book.clear()
        for price, size in levels:
            if size <= 0:
                book.pop(price, None)
            else:
                book[float(price)] = float(size)

    def update(self, record: dict[str, Any]) -> None:
        event_type = str(record.get('event_type') or record.get('type') or '').lower()
        bids = parse_levels(record.get('bids'), descending=True)
        asks = parse_levels(record.get('asks'), descending=False)
        is_snapshot = event_type == 'book_snapshot'

        if bids:
            self._apply_side('bids', bids, replace=is_snapshot)
        elif is_snapshot:
            self.bids.clear()
        if asks:
            self._apply_side('asks', asks, replace=is_snapshot)
        elif is_snapshot:
            self.asks.clear()

        for change in _safe_json(record.get('price_change')) or []:
            if isinstance(change, dict):
                side = str(change.get('side') or change.get('book') or '').lower()
                price = change.get('price') or change.get('px')
                size = change.get('size') or change.get('sz') or 0
            elif isinstance(change, (list, tuple)) and len(change) >= 3:
                side, price, size = change[0], change[1], change[2]
            else:
                continue
            try:
                price, size = float(price), float(size)
            except (TypeError, ValueError):
                continue
            target = self.bids if 'bid' in side else self.asks
            if size <= 0:
                target.pop(price, None)
            else:
                target[price] = size

    def best_bid(self) -> tuple[float | None, float]:
        if not self.bids:
            return None, 0.0
        price = max(self.bids)
        return price, float(self.bids[price])

    def best_ask(self) -> tuple[float | None, float]:
        if not self.asks:
            return None, 0.0
        price = min(self.asks)
        return price, float(self.asks[price])

    def vwap_buy(self, stake_usdc: float) -> tuple[float | None, float]:
        remaining, total_cost, total_shares = float(stake_usdc), 0.0, 0.0
        for price in sorted(self.asks):
            size = float(self.asks[price])
            if size <= 0:
                continue
            level_cost = float(price) * size
            if level_cost <= remaining:
                total_cost += level_cost
                total_shares += size
                remaining -= level_cost
            else:
                shares = remaining / float(price)
                total_cost += remaining
                total_shares += shares
                remaining = 0.0
                break
        if total_shares <= 0 or remaining > 1e-6:
            return None, total_shares
        return total_cost / total_shares, total_shares

@dataclass
class MarketPosition:
    market_id: str
    game_id: Any
    yes_shares: float = 0.0
    no_shares: float = 0.0
    yes_cost: float = 0.0
    no_cost: float = 0.0
    realized_pnl: float = 0.0
    cooldown_until: pd.Timestamp | None = None
    settled: bool = False
    settlement_value: float | None = None

@dataclass
class PortfolioState:
    initial_bankroll: float
    cash: float
    realized_pnl: float = 0.0
    positions: dict[str, MarketPosition] = field(default_factory=dict)

def total_bankroll(portfolio: PortfolioState) -> float:
    return float(portfolio.initial_bankroll + portfolio.realized_pnl)

def available_capital(portfolio: PortfolioState) -> float:
    return max(total_bankroll(portfolio) / PER_GAME_BANKROLL_DIVISOR, 0.0)

def fee_adjusted_kelly(probability: float, market_price: float, fee_rate: float = FEE_RATE, fraction: float = 0.25) -> float:
    if not (0 < market_price < 1) or not (0 < probability < 1):
        return 0.0
    b = (1.0 - market_price) / market_price
    q = 1.0 - probability
    full_kelly = ((probability * (1.0 - fee_rate) * b) - q) / b
    return round(max(full_kelly * fraction, 0.0), 6)

def ensure_position(portfolio: PortfolioState, market_id: str, game_id: Any) -> MarketPosition:
    if market_id not in portfolio.positions:
        portfolio.positions[market_id] = MarketPosition(market_id=market_id, game_id=game_id)
    return portfolio.positions[market_id]

def merge_positions(portfolio: PortfolioState, position: MarketPosition) -> None:
    paired = min(position.yes_shares, position.no_shares)
    if paired <= 0:
        return
    yes_release_cost = position.yes_cost * (paired / position.yes_shares) if position.yes_shares > 0 else 0.0
    no_release_cost = position.no_cost * (paired / position.no_shares) if position.no_shares > 0 else 0.0
    position.yes_shares -= paired
    position.no_shares -= paired
    position.yes_cost -= yes_release_cost
    position.no_cost -= no_release_cost
    realized = paired - yes_release_cost - no_release_cost
    portfolio.cash += paired
    portfolio.realized_pnl += realized
    position.realized_pnl += realized

def execute_buy(portfolio: PortfolioState, position: MarketPosition, book: OrderBookState, side: str, stake_usdc: float) -> dict[str, float] | None:
    if stake_usdc <= 0:
        return None
    spend = min(float(stake_usdc), float(portfolio.cash))
    if spend <= 0:
        return None
    raw_vwap, raw_shares = book.vwap_buy(spend)
    if raw_vwap is None or raw_shares <= 0:
        return None
    effective_price = min(max(float(raw_vwap) + ADVERSE_SELECTION_PENALTY, 0.001), 0.999)
    shares = spend / effective_price
    portfolio.cash -= spend
    if side == 'YES':
        position.yes_shares += shares
        position.yes_cost += spend
    else:
        position.no_shares += shares
        position.no_cost += spend
    merge_positions(portfolio, position)
    return {'spend': spend, 'price': effective_price, 'shares': shares}

def settle_position(portfolio: PortfolioState, position: MarketPosition, outcome_yes: bool) -> dict[str, float] | None:
    if position.settled:
        return None
    winning_shares = position.yes_shares if outcome_yes else position.no_shares
    total_cost = position.yes_cost + position.no_cost
    proceeds = float(winning_shares)
    gross_profit = proceeds - total_cost
    fee = max(gross_profit, 0.0) * FEE_RATE
    net_profit = gross_profit - fee
    portfolio.cash += proceeds - fee
    portfolio.realized_pnl += net_profit
    position.realized_pnl += net_profit
    position.yes_shares, position.no_shares, position.yes_cost, position.no_cost = 0.0, 0.0, 0.0, 0.0
    position.settled = True
    position.settlement_value = 1.0 if outcome_yes else 0.0
    return {'proceeds': proceeds, 'gross_profit': gross_profit, 'fee': fee, 'net_profit': net_profit}

In [ ]:
def normalize_pmxt_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c) for c in out.columns]

    # PMXT schema: timestamp_created_at, market_id (hex condition ID),
    # update_type, data (JSON string containing token_id, side, bids, asks, etc.)
    has_data_col = 'data' in out.columns

    # --- Timestamp ---
    if 'timestamp_created_at' in out.columns:
        out['timestamp'] = pd.to_datetime(out['timestamp_created_at'], utc=True, errors='coerce')
    elif 'timestamp_received' in out.columns:
        out['timestamp'] = pd.to_datetime(out['timestamp_received'], utc=True, errors='coerce')
    else:
        timestamp_col = _candidate_column(list(out.columns), ['timestamp', 'event_time', 'ts', 'created_at'])
        if timestamp_col is None:
            raise ValueError('PMXT chunk is missing a timestamp column')
        out['timestamp'] = pd.to_datetime(out[timestamp_col], utc=True, errors='coerce')

    # --- event_type from top-level update_type ---
    if 'update_type' in out.columns:
        out['event_type'] = out['update_type'].astype(str)
    elif 'event_type' not in out.columns:
        out['event_type'] = 'price_change'

    # --- Parse the `data` JSON column and hoist all subfields ---
    if has_data_col:
        def _parse_data_row(raw):
            if isinstance(raw, dict):
                return raw
            if isinstance(raw, str) and raw.strip():
                try:
                    return json.loads(raw)
                except Exception:
                    pass
            return {}

        parsed = out['data'].apply(_parse_data_row)

        out['token_id'] = parsed.apply(lambda d: str(d.get('token_id', '')) if d.get('token_id') is not None else None)
        out['side'] = parsed.apply(lambda d: str(d.get('side', '')).upper())
        out['bids'] = parsed.apply(lambda d: d.get('bids'))
        out['asks'] = parsed.apply(lambda d: d.get('asks'))
        out['best_bid'] = parsed.apply(lambda d: d.get('best_bid'))
        out['best_ask'] = parsed.apply(lambda d: d.get('best_ask'))
        # price_change rows carry change_price/change_size/change_side
        out['price_change'] = parsed.apply(lambda d: (
            [{'side': d.get('change_side', '').lower(), 'price': d.get('change_price'), 'size': d.get('change_size')}]
            if d.get('change_price') is not None else None
        ))
        out['resolution'] = None
        # If update_type wasn't a top-level column, fall back to data payload
        if 'update_type' not in frame.columns:
            out['event_type'] = parsed.apply(lambda d: str(d.get('update_type', 'price_change')))
    else:
        # Non-PMXT schema: use legacy candidate-column path
        market_col = _candidate_column(list(out.columns), ['token_id', 'asset_id', 'market_id', 'market', 'slug', 'event_slug'])
        if market_col is None:
            raise ValueError('PMXT chunk is missing a market identifier column')
        out['token_id'] = out[market_col].astype(str)
        for col in ['bids', 'asks', 'price_change', 'resolution']:
            if col not in out.columns:
                out[col] = None

    # Ensure market_id column exists (use hex condition ID from top level if present)
    if 'market_id' not in out.columns:
        out['market_id'] = out.get('token_id', pd.Series([''] * len(out)))

    return out


def iter_pmxt_chunks(paths: list[Path], chunksize: int = CHUNK_SIZE):
    for path in tqdm(paths, desc='PMXT parquet files'):
        frame = pl.read_parquet(path)
        total_rows = frame.height
        for offset in range(0, total_rows, chunksize):
            yield path, frame.slice(offset, chunksize).to_pandas()


def _sample_values(values, limit: int = 8) -> list[str]:
    if isinstance(values, pd.Series):
        iterable = values.dropna().astype(str).tolist()
    else:
        iterable = [str(v) for v in values if v is not None and str(v) != '']
    seen, seen_set = [], set()
    for value in iterable:
        if value in seen_set:
            continue
        seen.append(value)
        seen_set.add(value)
        if len(seen) >= limit:
            break
    return seen


def _chunk_key_diagnostics(frame: pd.DataFrame, market_lookup: pd.DataFrame, sample_limit: int = 8) -> dict[str, Any]:
    lookup_keys = set(market_lookup['lookup_key'].dropna().astype(str)) if 'lookup_key' in market_lookup.columns else set()
    candidate_cols = [col for col in ['token_id', 'asset_id', 'market_id', 'slug', 'event_slug', 'market'] if col in frame.columns]
    key_stats = []
    for col in candidate_cols:
        values = frame[col].dropna().astype(str)
        unique_values = pd.Index(values.unique())
        lookup_matches = unique_values.isin(lookup_keys)
        unmatched_values = unique_values[~lookup_matches]
        key_stats.append({
            'column': col,
            'rows_non_null': int(values.shape[0]),
            'unique_values': int(len(unique_values)),
            'lookup_key_matches': int(lookup_matches.sum()),
            'sample_unmatched': _sample_values(list(unmatched_values), limit=sample_limit),
        })
    selected_key_col = _candidate_column(list(frame.columns), ['token_id', 'asset_id', 'market_id', 'slug', 'event_slug', 'market'])
    selected_unmatched = []
    if selected_key_col is not None:
        selected_values = frame[selected_key_col].dropna().astype(str)
        matched_mask = selected_values.isin(lookup_keys)
        selected_unmatched = _sample_values(selected_values[~matched_mask], limit=sample_limit)
    return {
        'selected_key_col': selected_key_col,
        'key_stats': key_stats,
        'selected_unmatched': selected_unmatched,
    }


def _print_chunk_diagnostics(path: Path, chunk_index: int, raw_chunk: pd.DataFrame, normalized: pd.DataFrame, attached: pd.DataFrame, joined: pd.DataFrame, market_lookup: pd.DataFrame, timeline: pd.DataFrame) -> None:
    diag = _chunk_key_diagnostics(normalized, market_lookup)
    ts_min = normalized['timestamp'].min() if 'timestamp' in normalized.columns and not normalized.empty else pd.NaT
    ts_max = normalized['timestamp'].max() if 'timestamp' in normalized.columns and not normalized.empty else pd.NaT
    tipoff_min = timeline['tipoff_time'].min() if 'tipoff_time' in timeline.columns and not timeline.empty else pd.NaT
    tipoff_max = timeline['tipoff_time'].max() if 'tipoff_time' in timeline.columns and not timeline.empty else pd.NaT
    joined_ts_min = joined['timestamp'].min() if 'timestamp' in joined.columns and not joined.empty else pd.NaT
    joined_ts_max = joined['timestamp'].max() if 'timestamp' in joined.columns and not joined.empty else pd.NaT

    print(f'Chunk #{chunk_index} | {path.name}')
    print(f"  rows raw={len(raw_chunk)} normalized={len(normalized)} attached={len(attached)} joined={len(joined)}")
    print(f"  normalized ts: {ts_min} -> {ts_max}")
    print(f"  timeline tipoff: {tipoff_min} -> {tipoff_max}")
    print(f"  joined ts: {joined_ts_min} -> {joined_ts_max}")
    print(f"  selected key col: {diag['selected_key_col']}")
    if diag['selected_unmatched']:
        print(f"  sample unmatched keys: {diag['selected_unmatched']}")
    for stat in diag['key_stats']:
        print(f"  col={stat['column']} non_null={stat['rows_non_null']} unique={stat['unique_values']} matches={stat['lookup_key_matches']}")
        if stat['sample_unmatched']:
            print(f"    unmatched samples: {stat['sample_unmatched']}")


def _extract_resolution(record: dict[str, Any]) -> bool | None:
    event_type = str(record.get('event_type') or '').lower()
    if event_type not in {'condition_resolved', 'market_resolution'} and record.get('resolution') in (None, '', {}, []):
        return None
    resolution = record.get('resolution')
    payload = _safe_json(resolution) if not isinstance(resolution, dict) else resolution
    winner = str(payload.get('winner') or payload.get('outcome') or payload.get('result') or '').lower() if isinstance(payload, dict) else str(resolution).lower()
    if winner in {'yes', '1', 'true', 'home'}:
        return True
    if winner in {'no', '0', 'false', 'away'}:
        return False
    return None


def evaluate_signal(probability_yes: float, position: MarketPosition, portfolio: PortfolioState, book: OrderBookState) -> tuple[str, float] | None:
    best_ask, ask_size = book.best_ask()
    best_bid, bid_size = book.best_bid()
    if best_ask is None or best_bid is None:
        return None

    if float(best_ask) * float(ask_size) < MIN_RESTING_DEPTH_USDC or float(best_bid) * float(bid_size) < MIN_RESTING_DEPTH_USDC:
        return None

    yes_fraction = fee_adjusted_kelly(probability_yes, float(best_ask))
    no_fraction = fee_adjusted_kelly(1.0 - probability_yes, max(min(1.0 - float(best_bid), 0.999), 0.001))

    if yes_fraction <= 0 and no_fraction <= 0:
        return None

    if yes_fraction >= no_fraction:
        target = available_capital(portfolio) * yes_fraction
        delta = target - position.yes_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'YES', float(delta)
    else:
        target = available_capital(portfolio) * no_fraction
        delta = target - position.no_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'NO', float(delta)
    return None


def run_backtest(paths: list[Path], timeline: pd.DataFrame, market_lookup: pd.DataFrame, model: Any, feature_cols: list[str] | None, initial_bankroll: float = INITIAL_BANKROLL) -> dict[str, Any]:
    portfolio = PortfolioState(initial_bankroll=initial_bankroll, cash=initial_bankroll)
    books: dict[str, OrderBookState] = {}
    trades, equity = [], []
    stage_counts = {
        'chunks_seen': 0,
        'raw_rows': 0,
        'normalized_rows': 0,
        'attached_rows': 0,
        'joined_rows': 0,
        'positions_created': 0,
        'signals_triggered': 0,
        'fills_executed': 0,
        'settlements': 0,
    }
    diagnostics_printed = 0
    preview_limit = 5

    print(f'Backtest start: lookup_rows={len(market_lookup)} timeline_rows={len(timeline)} unique_lookup_keys={market_lookup["lookup_key"].nunique() if "lookup_key" in market_lookup.columns else 0}')
    if 'lookup_key' in market_lookup.columns:
        print(f"  sample lookup keys: {_sample_values(market_lookup['lookup_key'], limit=6)}")

    for path, chunk in iter_pmxt_chunks(paths):
        stage_counts['chunks_seen'] += 1
        raw_chunk = chunk.copy()
        normalized = normalize_pmxt_frame(raw_chunk)
        attached = attach_game_ids(normalized, market_lookup)
        joined = join_ticks_with_timeline(attached, timeline)

        stage_counts['raw_rows'] += len(raw_chunk)
        stage_counts['normalized_rows'] += len(normalized)
        stage_counts['attached_rows'] += len(attached)
        stage_counts['joined_rows'] += len(joined)

        should_print = (diagnostics_printed < preview_limit) or \
                       (len(attached) == 0 and diagnostics_printed < preview_limit + 3) or \
                       (len(joined) == 0 and len(attached) > 0 and diagnostics_printed < preview_limit + 3)
        if should_print:
            _print_chunk_diagnostics(path, stage_counts['chunks_seen'], raw_chunk, normalized, attached, joined, market_lookup, timeline)
            diagnostics_printed += 1

        for record in joined.sort_values('timestamp').to_dict('records'):
            market_id = str(record.get('resolved_market_id') or record['market_id'])
            if market_id not in portfolio.positions:
                stage_counts['positions_created'] += 1
            position = ensure_position(portfolio, market_id, record['game_id'])
            book = books.setdefault(market_id, OrderBookState())
            book.update(record)

            resolution = _extract_resolution(record)
            if resolution is not None:
                settlement = settle_position(portfolio, position, resolution)
                if settlement:
                    stage_counts['settlements'] += 1
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': 'SETTLE_YES' if resolution else 'SETTLE_NO', **settlement})
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.settled or (record.get('time_remaining') is not None and float(record['time_remaining']) <= 0):
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.cooldown_until is not None and pd.Timestamp(record['timestamp']) < position.cooldown_until:
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            adv_ctx = None
            if USE_ADVANCED_FEATURES:
                adv_ctx = {
                    'starttype_encoded': int(record.get('starttype_encoded', STARTTYPE_FALLBACK) or STARTTYPE_FALLBACK),
                    'player_quality_home': float(record.get('player_quality_home', 0.0) or 0.0),
                    'player_quality_away': float(record.get('player_quality_away', 0.0) or 0.0),
                }

            prob_home = predict_home_win_prob(
                model=model,
                feature_cols=feature_cols,
                home_score=int(record['home_score']),
                away_score=int(record['away_score']),
                period=int(record['period']),
                clock=str(record['clock']),
                advanced_ctx=adv_ctx,
            )
            yes_team_side = str(record.get('yes_team_side') or 'home').lower()
            prob_yes = prob_home if yes_team_side == 'home' else 1.0 - prob_home

            signal = evaluate_signal(prob_yes, position, portfolio, book)
            if signal:
                stage_counts['signals_triggered'] += 1
                side, delta = signal
                fill = execute_buy(portfolio, position, book, side, delta)
                if fill:
                    stage_counts['fills_executed'] += 1
                    position.cooldown_until = pd.Timestamp(record['timestamp']) + pd.to_timedelta(COOLDOWN_MS, unit='ms')
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': f'BUY_{side}', 'model_prob_home': prob_home, 'model_prob_yes': prob_yes, 'yes_team_side': yes_team_side, 'cash_after': portfolio.cash, **fill})
            equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})

    print('Backtest stage totals:')
    for key, val in stage_counts.items():
        print(f'  {key}: {val}')
    if stage_counts['raw_rows'] > 0:
        print(f"  attach_rate: {stage_counts['attached_rows'] / stage_counts['raw_rows']:.4%}")
        print(f"  join_rate:   {stage_counts['joined_rows'] / stage_counts['raw_rows']:.4%}")
    if stage_counts['attached_rows'] > 0:
        print(f"  post_attach_join_rate: {stage_counts['joined_rows'] / stage_counts['attached_rows']:.4%}")

    BACKTEST_STATE_PATH.write_text(json.dumps({
        'initial_bankroll': portfolio.initial_bankroll,
        'cash': portfolio.cash,
        'realized_pnl': portfolio.realized_pnl,
        'positions': {m: asdict(p) for m, p in portfolio.positions.items()},
    }, default=str, indent=2), encoding='utf-8')
    return {'portfolio': portfolio, 'trades': pd.DataFrame(trades), 'equity_curve': pd.DataFrame(equity), 'state_path': str(BACKTEST_STATE_PATH), 'stage_counts': stage_counts}


def plot_equity_curve(result: dict[str, Any], label: str) -> None:
    equity = result['equity_curve'].copy()
    if equity.empty:
        return
    sampled = equity.iloc[::max(len(equity) // min(2000, len(equity)), 1)].copy()
    plt.figure(figsize=(12, 4))
    plt.plot(sampled['timestamp'], sampled['equity'])
    plt.title(f'Equity Curve — {label}')
    plt.xlabel('Timestamp')
    plt.ylabel('USDC')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
if not MARKET_ONLY_MODE:
    raise NotImplementedError('This Modal notebook currently preserves the market-only path from the parquet Colab notebook.')

model, feature_cols = load_model(str(MODEL_PATH))
if model is None:
    raise FileNotFoundError(f'Model not found at {MODEL_PATH}')

pmxt_paths = sorted(PMXT_ROOT.glob(PMXT_GLOB))
if not pmxt_paths:
    raise FileNotFoundError(f'No PMXT files found under {PMXT_ROOT} matching {PMXT_GLOB}')

pmxt_start_date, pmxt_end_date = infer_pmxt_date_range(pmxt_paths)
print(f'PMXT date range: {pmxt_start_date} -> {pmxt_end_date}')
print('Running in MARKET-ONLY mode - using Gamma API as game catalog (no NBA stats required)')

gamma_markets_df = search_nba_markets(pmxt_start_date, pmxt_end_date, limit=100, closed=True, active=True, buffer_days=MARKET_DATE_BUFFER_DAYS)
if gamma_markets_df.empty:
    raise ValueError(f'No Gamma NBA markets found between {pmxt_start_date} and {pmxt_end_date}')

print(f'Found {len(gamma_markets_df)} Gamma markets')
game_catalog = build_game_catalog_from_gamma(gamma_markets_df)
if game_catalog.empty:
    raise ValueError('Could not build game catalog from Gamma markets')

print(f'Built game catalog with {len(game_catalog)} entries')
timeline = build_simple_timeline_from_gamma(gamma_markets_df)
if timeline.empty:
    raise ValueError('Could not build timeline from Gamma markets')

print(f'Built timeline with {len(timeline)} entries')

# Build lookup keyed by CLOB token IDs (what PMXT data/token_id contains)
# Each Gamma market has clobTokenIds: [yes_token_id, no_token_id]
# PMXT rows carry token_id (inside data JSON) — one per YES/NO side per row
lookup_rows = []
token_id_count = 0
for row in game_catalog.to_dict('records'):
    clob_token_ids = row.get('clob_token_ids') or []
    yes_team_side = row.get('yes_team_side', 'home')
    for i, token_id in enumerate(clob_token_ids):
        if not token_id:
            continue
        # clobTokenIds[0] = YES token, clobTokenIds[1] = NO token
        # yes_team_side already reflects which team the YES outcome is for
        lookup_rows.append({
            'lookup_key': str(token_id),
            'game_id': row['game_id'],
            'yes_team_side': yes_team_side,
            'market_id': row['market_id'],
            'token_side': 'YES' if i == 0 else 'NO',
        })
        token_id_count += 1

print(f'Token ID lookup entries: {token_id_count} (from {len(game_catalog)} catalog entries)')
if token_id_count == 0:
    print('WARNING: No CLOB token IDs found in game catalog. Falling back to integer market_id keys (likely to produce zero matches).')
    for row in game_catalog.to_dict('records'):
        lookup_rows.append({
            'lookup_key': str(row['market_id']),
            'game_id': row['game_id'],
            'yes_team_side': row.get('yes_team_side', 'home'),
            'market_id': row['market_id'],
            'token_side': 'YES',
        })

MARKET_LOOKUP = pd.DataFrame(lookup_rows)
MARKET_MATCHES = game_catalog.copy()
MARKET_MATCHES['score'] = 10.0
MARKET_MATCHES['score_gap'] = 5.0
MARKET_MATCHES['time_delta_hours'] = 0.0

print(f'Market lookup has {len(MARKET_LOOKUP)} entries')
display_cols = ['game_date', 'game_id', 'home_team', 'away_team', 'market_id', 'event_title', 'question', 'yes_team_side']
display(MARKET_MATCHES[display_cols].head(20))

print(f'Beginning Backtest Iteration on {len(pmxt_paths)} files')
tuning_result = run_backtest(pmxt_paths, timeline, MARKET_LOOKUP, model, feature_cols, initial_bankroll=INITIAL_BANKROLL)

print(f"Ending Balance: {tuning_result['portfolio'].cash:.2f}")
print(f"State saved to: {tuning_result['state_path']}")
if not tuning_result['trades'].empty:
    display(tuning_result['trades'].tail(20))
plot_equity_curve(tuning_result, 'full')